In [1]:
# Load frozen validation choices without retraining or choosing on proof
from pathlib import Path
import sys
import numpy as np
import pandas as pd

ROOT = next(folder for folder in (Path.cwd(), *Path.cwd().parents)
            if (folder / 'src' / 'backtest.py').is_file())
sys.path.insert(0, str(ROOT / 'src'))
import contract
import backtest
from backtest import lookup_prob, k_of

globals().update(backtest.load_frozen_state())


ModuleNotFoundError: No module named 'features'

# Proof on 2017 (run once)

Models and settings are frozen. 2017 is only used for scoring, never for choosing.

In [ ]:
# Test probabilities and nightly tables for every model
P_test = {'Lookup table': lookup_prob(test_bookings)}
P_test.update({n: np.clip(m.predict_proba(X_test[final_features])[:, 1], 0, 1) for n, m in models.items()})
N_test = {n: contract.nightly_table(test_bookings, p) for n, p in P_test.items()}

In [ ]:
# Score every frozen policy on 2017 with the contract
rows = []
for r in best.itertuples():
    n = N_test.get(r.model, N_test['Lookup table'])
    rows.append({'model': r.model, 'rule': r.rule, 'value': r.value, **contract.night_cost(n, k_of(n, r.rule, r.value))})
n0 = N_test['Lookup table']
rows.append({'model': 'No oversell', 'rule': 'k=0', 'value': 0, **contract.night_cost(n0, pd.Series(0, index=n0.index))})
proof = pd.DataFrame(rows)
proof[['model', 'rule', 'value', 'cost_eur', 'empty_rooms', 'walked_guests', 'walks_per_1000', 'ceiling_met']]

In [ ]:
# Result of the selected policy against the client flat rule
sel = proof[(proof['model'] == selected['model']) & (proof['rule'] == selected['rule'])].iloc[0]
flat = proof[proof['rule'] == 'flat'].iloc[0]
saving = flat['cost_eur'] - sel['cost_eur']
print('Selected:', sel['model'], sel['rule'], sel['value'], '| cost:', sel['cost_eur'], '| walks per 1,000:', sel['walks_per_1000'])
print('Savings vs flat rule (EUR):', round(saving, 2), '|', round(100 * saving / flat['cost_eur'], 2), '% | per month:', round(saving / 8, 2))

In [ ]:
# Final decision wording follows the selected policy
if sel['ceiling_met']:
    print('Recommendation: use', sel['model'], '/', sel['rule'], '(safe on 2017 data)')
else:
    print('Warning: the selected policy breaks the walk ceiling on 2017 data. Do not call it safe.')

# 14-Night Recommendation Table

Built with the **selected, frozen policy** (nights 2017-08-18 to 2017-08-31, both hotels). The `cancelled` column is not used.

In [ ]:
# Nightly table for the 14 nights, using the selected policy's probabilities
nights = pd.date_range('2017-08-18', periods=14)
mask = test_bookings['arrival_date'].isin(nights).to_numpy()
src = selected['model'] if selected['model'] in P_test else 'Lookup table'
n14 = contract.nightly_table(test_bookings[mask], P_test[src][mask])

In [ ]:
# Recommended k with the selected rule
n14['k'] = k_of(n14, selected['rule'], selected['value'])
table = n14[['book', 'expected', 'spread', 'k']].round(1).reset_index()
table.columns = ['hotel', 'night', 'bookings', 'expected_cancellations', 'spread', 'recommended_k']
table.insert(2, 'policy', f"{selected['model']} / {selected['rule']} = {selected['value']}")
table = table.sort_values(['night', 'hotel']).reset_index(drop=True)
table

In [ ]:
# Save the table inside the project folder
(ROOT / 'evidence').mkdir(exist_ok=True)
table.to_csv(ROOT / 'evidence' / 'nightly_k.csv', index=False)

# Limitations

- No room capacity: k is measured beyond each night's bookings, as the contract scores it.
- Live nights that are still filling have incomplete bookings, so their k is provisional.
- Cancellation is not a no-show.
- Summed probabilities assume independent bookings; real cancellations can be correlated.
- The walk ceiling is an average over all nights; single nights can still walk guests.
- Models and settings were tuned on one half-year of validation data, so validation scores are slightly optimistic.
- Feature eligibility is a team decision (see the Feature Eligibility section).
- Past cancellation behaviour may change.